In [128]:
from pathlib import Path
import numpy as np
import pandas as pd


In [129]:
RESULT_DIRS = [
    Path("../results/cifar10/no_attacks"),
    Path("../results/cifar10/classic_attacks"),
    Path("../results/cifar10/pca_attacks"),
]

AGGREGATORS = ["fedavg", "median", "trimmedmean", "krum", "multikrum"]
SEEDS = [40, 60, 11]

MAX_ROUND = 50
FULL_ROUNDS = pd.DataFrame({"round": range(MAX_ROUND + 1)})
RANDOM_GUESS_ACC = 0.1
LAST_N = 5

DIAGNOSTIC_METRICS = [
    "malicious_to_benign_mean_distance",
    "malicious_to_benign_mean_cosine",
    "attack_aggregation_shift",
]


In [130]:
DEFENSE_LABELS = {
    "fedavg": "FedAvg",
    "median": "Median",
    "trimmedmean": "TrimmedMean",
    "krum": "Krum",
    "multikrum": "MultiKrum",
}

ATTACK_LABELS = {
    "no_attack": "No Attack",
    "alie_optim": "ALIE",
    "min_max": "MinMax",
    "min_sum": "MinSum",
    "sign_flip": "SF",
    "stdk2_p0": "PCA k=2",
    "stdk3_p0": "PCA k=3",
}

defense_order = ["FedAvg", "Median", "TrimmedMean", "Krum", "MultiKrum"]
attack_order = ["No Attack", "ALIE", "MinMax", "MinSum", "SF", "PCA k=2", "PCA k=3"]


In [131]:
def parse_result_name(path):
    stem = path.stem
    source_dir = path.parent.name

    if source_dir == "no_attacks":
        aggregator, nnm_mode, seed = stem.rsplit("_", 2)
        return aggregator, "no_attack", nnm_mode, int(seed)

    if source_dir in {"classic_attacks", "pca_attacks"}:
        prefix, nnm_mode, seed = stem.rsplit("_", 2)
        for aggregator in AGGREGATORS:
            marker = f"{aggregator}_"
            if prefix.startswith(marker):
                attack = prefix[len(marker):]
                return aggregator, attack, nnm_mode, int(seed)

    raise ValueError(f"Cannot parse result filename: {path}")


In [132]:
all_runs = []

for result_dir in RESULT_DIRS:
    if not result_dir.exists():
        print(f"Missing directory: {result_dir}")
        continue

    for path in sorted(result_dir.glob("*.csv")):
        aggregator, attack, nnm_mode, seed = parse_result_name(path)

        if aggregator not in AGGREGATORS or seed not in SEEDS:
            continue

        df = pd.read_csv(path)
        df["round"] = pd.to_numeric(df["round"], errors="coerce")
        df["test_accuracy"] = pd.to_numeric(df["test_accuracy"], errors="coerce")

        keep_cols = ["round", "test_accuracy"]
        for metric in DIAGNOSTIC_METRICS:
            if metric in df.columns:
                df[metric] = pd.to_numeric(df[metric], errors="coerce")
            else:
                df[metric] = np.nan
            keep_cols.append(metric)

        df = df[keep_cols].dropna(subset=["round"])
        df["round"] = df["round"].astype(int)

        df = FULL_ROUNDS.merge(df, on="round", how="left")
        df["test_accuracy_filled"] = df["test_accuracy"].fillna(RANDOM_GUESS_ACC)
        df["aggregator"] = aggregator
        df["attack"] = attack
        df["nnm_mode"] = nnm_mode
        df["seed"] = seed
        df["file"] = path.name
        df["source_dir"] = result_dir.name
        all_runs.append(df)

all_results = pd.concat(all_runs, ignore_index=True)
all_results.head()


,round,test_accuracy,malicious_to_benign_mean_distance,malicious_to_benign_mean_cosine,attack_aggregation_shift,test_accuracy_filled,aggregator,attack,nnm_mode,seed,file,source_dir
0,0,0.1007,NaN,NaN,NaN,0.1007,fedavg,no_attack,nnm,11,fedavg_nnm_11.csv,no_attacks
1,1,0.1450,NaN,NaN,NaN,0.1450,fedavg,no_attack,nnm,11,fedavg_nnm_11.csv,no_attacks
2,2,0.1650,NaN,NaN,NaN,0.1650,fedavg,no_attack,nnm,11,fedavg_nnm_11.csv,no_attacks
3,3,0.1955,NaN,NaN,NaN,0.1955,fedavg,no_attack,nnm,11,fedavg_nnm_11.csv,no_attacks
4,4,0.2395,NaN,NaN,NaN,0.2395,fedavg,no_attack,nnm,11,fedavg_nnm_11.csv,no_attacks


In [133]:
SUMMARY_METRICS = ["test_accuracy_filled", *DIAGNOSTIC_METRICS]

run_summary = (
    all_results
    .sort_values(["aggregator", "attack", "nnm_mode", "seed", "round"])
    .groupby(["aggregator", "attack", "nnm_mode", "seed"], as_index=False)
    .tail(LAST_N)
    .groupby(["aggregator", "attack", "nnm_mode", "seed"], as_index=False)
    [SUMMARY_METRICS]
    .mean()
)

avg_results = (
    run_summary
    .groupby(["aggregator", "attack", "nnm_mode"], as_index=False)
    [SUMMARY_METRICS]
    .agg(["mean", "std", "count"])
)

avg_results.columns = [
    "_".join(col).strip("_")
    for col in avg_results.columns.to_flat_index()
]
avg_results = avg_results.reset_index()

avg_results["test_accuracy_filled_mean"] *= 100
avg_results["test_accuracy_filled_std"] = avg_results["test_accuracy_filled_std"].fillna(0) * 100
for metric in DIAGNOSTIC_METRICS:
    avg_results[f"{metric}_std"] = avg_results[f"{metric}_std"].fillna(0)

avg_results["accuracy_cell"] = avg_results.apply(
    lambda r: f'{r["test_accuracy_filled_mean"]:.2f} ± {r["test_accuracy_filled_std"]:.2f}',
    axis=1,
)
avg_results["defense"] = avg_results["aggregator"].map(DEFENSE_LABELS).fillna(avg_results["aggregator"])
avg_results["attack_label"] = avg_results["attack"].map(ATTACK_LABELS).fillna(avg_results["attack"])

avg_results.head()


,index,aggregator,attack,nnm_mode,test_accuracy_filled_mean,test_accuracy_filled_std,test_accuracy_filled_count,malicious_to_benign_mean_distance_mean,malicious_to_benign_mean_distance_std,malicious_to_benign_mean_distance_count,malicious_to_benign_mean_cosine_mean,malicious_to_benign_mean_cosine_std,malicious_to_benign_mean_cosine_count,attack_aggregation_shift_mean,attack_aggregation_shift_std,attack_aggregation_shift_count,accuracy_cell,defense,attack_label
0,0,fedavg,alie_optim,nnm,48.943333,6.146619,3,0.564017,0.136053,3,-0.560232,0.048508,3,0.230222,0.067555,3,48.94 ± 6.15,FedAvg,ALIE
1,1,fedavg,alie_optim,raw,40.329333,10.855078,3,1.003491,0.138111,3,-0.694418,0.069670,3,0.301047,0.041433,3,40.33 ± 10.86,FedAvg,ALIE
2,2,fedavg,min_max,nnm,36.390667,22.993058,3,0.759619,0.114674,2,-0.243559,0.091980,2,0.144557,0.047029,2,36.39 ± 22.99,FedAvg,MinMax
3,3,fedavg,min_max,raw,21.334667,19.632219,3,0.895425,0.000000,1,-0.192209,0.000000,1,0.268613,0.000000,1,21.33 ± 19.63,FedAvg,MinMax
4,4,fedavg,min_sum,nnm,34.188000,21.086059,3,0.580313,0.237921,2,-0.100172,0.029449,2,0.186909,0.020021,2,34.19 ± 21.09,FedAvg,MinSum


In [134]:
def make_accuracy_table(avg_results, nnm_mode):
    mode_df = avg_results[avg_results["nnm_mode"] == nnm_mode].copy()

    present_defenses = [
        defense for defense in defense_order
        if defense in set(mode_df["defense"])
    ]
    present_attacks = [
        attack for attack in attack_order
        if attack in set(mode_df["attack_label"])
    ]

    return (
        mode_df
        .pivot(index="defense", columns="attack_label", values="accuracy_cell")
        .reindex(index=present_defenses, columns=present_attacks)
        .rename_axis(index="Defense", columns=None)
    )


In [135]:
raw_accuracy_table = make_accuracy_table(avg_results, "raw")
raw_accuracy_table


,No Attack,ALIE,MinMax,MinSum,SF,PCA k=2,PCA k=3
Defense,,,,,,,
FedAvg,64.31 ± 1.26,40.33 ± 10.86,21.33 ± 19.63,36.82 ± 23.28,49.90 ± 0.94,49.17 ± 3.20,49.55 ± 3.81
Median,55.94 ± 3.11,21.29 ± 7.64,39.02 ± 2.78,41.52 ± 3.41,23.56 ± 1.25,33.82 ± 6.65,34.50 ± 2.86
TrimmedMean,64.34 ± 1.29,15.37 ± 4.94,38.72 ± 3.13,40.37 ± 3.99,21.38 ± 3.97,32.29 ± 2.19,27.44 ± 7.94
Krum,25.26 ± 17.69,10.00 ± 0.00,10.00 ± 0.00,9.99 ± 0.01,19.06 ± 15.69,10.00 ± 0.00,10.00 ± 0.00
MultiKrum,61.01 ± 1.60,13.79 ± 6.57,18.29 ± 14.37,10.00 ± 0.00,10.00 ± 0.00,10.00 ± 0.00,10.00 ± 0.00


In [136]:
nnm_accuracy_table = make_accuracy_table(avg_results, "nnm")
nnm_accuracy_table


,No Attack,ALIE,MinMax,MinSum,SF,PCA k=2,PCA k=3
Defense,,,,,,,
FedAvg,64.28 ± 1.22,48.94 ± 6.15,36.39 ± 22.99,34.19 ± 21.09,40.14 ± 7.15,47.94 ± 2.94,46.36 ± 5.10
Median,64.32 ± 1.32,37.29 ± 12.17,39.91 ± 25.91,34.78 ± 21.48,10.00 ± 0.00,29.14 ± 13.82,31.04 ± 2.27
TrimmedMean,64.33 ± 1.30,38.50 ± 11.30,40.54 ± 26.46,24.68 ± 16.44,10.00 ± 0.00,22.58 ± 9.04,21.92 ± 6.95
Krum,64.31 ± 1.27,35.56 ± 11.29,39.80 ± 25.82,23.26 ± 22.98,11.58 ± 2.74,18.44 ± 6.74,16.96 ± 6.12
MultiKrum,64.29 ± 1.25,41.21 ± 11.42,39.89 ± 25.89,20.78 ± 18.67,10.00 ± 0.00,27.16 ± 6.43,18.34 ± 7.28


In [137]:
def make_diagnostic_table(avg_results, metric, nnm_mode):
    mode_df = avg_results[avg_results["nnm_mode"] == nnm_mode].copy()
    mean_col = f"{metric}_mean"
    std_col = f"{metric}_std"

    mode_df["diagnostic_cell"] = mode_df.apply(
        lambda r: "" if pd.isna(r[mean_col]) else f'{r[mean_col]:.4f} ± {r[std_col]:.4f}',
        axis=1,
    )

    present_defenses = [
        defense for defense in defense_order
        if defense in set(mode_df["defense"])
    ]
    present_attacks = [
        attack for attack in attack_order
        if attack in set(mode_df["attack_label"])
    ]

    return (
        mode_df
        .pivot(index="defense", columns="attack_label", values="diagnostic_cell")
        .reindex(index=present_defenses, columns=present_attacks)
        .rename_axis(index="Defense", columns=None)
    )


In [138]:
raw_malicious_to_benign_mean_distance_table = make_diagnostic_table(avg_results, "malicious_to_benign_mean_distance", "raw")
raw_malicious_to_benign_mean_distance_table


,No Attack,ALIE,MinMax,MinSum,SF,PCA k=2,PCA k=3
Defense,,,,,,,
FedAvg,,1.0035 ± 0.1381,0.8954 ± 0.0000,0.5055 ± 0.1430,0.1264 ± 0.0084,0.5999 ± 0.0684,0.6012 ± 0.0946
Median,,0.7061 ± 0.0813,0.8037 ± 0.1007,0.5810 ± 0.1539,0.7953 ± 0.0385,0.6368 ± 0.1082,0.5762 ± 0.0648
TrimmedMean,,0.5125 ± 0.1170,0.8269 ± 0.0695,0.5933 ± 0.1595,0.8076 ± 0.0173,0.6618 ± 0.1012,0.6004 ± 0.1082
Krum,,0.3350 ± 0.1858,0.4844 ± 0.1126,0.5391 ± 0.4256,1.0852 ± 0.3521,0.2696 ± 0.0000,0.2963 ± 0.0533
MultiKrum,,0.4510 ± 0.1097,0.5474 ± 0.3308,,2.3054 ± 2.8069,0.2881 ± 0.0173,0.3022 ± 0.0551


In [139]:
nnm_malicious_to_benign_mean_distance_table = make_diagnostic_table(avg_results, "malicious_to_benign_mean_distance", "nnm")
nnm_malicious_to_benign_mean_distance_table


,No Attack,ALIE,MinMax,MinSum,SF,PCA k=2,PCA k=3
Defense,,,,,,,
FedAvg,,0.5640 ± 0.1361,0.7596 ± 0.1147,0.5803 ± 0.2379,0.7974 ± 0.2423,0.6059 ± 0.0685,0.5908 ± 0.0857
Median,,0.7358 ± 0.1261,0.7583 ± 0.1526,0.5702 ± 0.2747,,0.5393 ± 0.0493,0.5778 ± 0.1020
TrimmedMean,,0.7441 ± 0.1584,0.7403 ± 0.1457,0.6609 ± 0.1013,,0.5291 ± 0.0729,0.5390 ± 0.1474
Krum,,0.7037 ± 0.0383,0.7407 ± 0.1260,0.7749 ± 0.0000,1.9857 ± 0.0000,0.4751 ± 0.1635,0.6189 ± 0.1026
MultiKrum,,0.7477 ± 0.0933,0.6533 ± 0.0045,0.7383 ± 0.0000,0.9438 ± 0.0000,0.6473 ± 0.1230,0.5503 ± 0.1147


In [140]:
raw_malicious_to_benign_mean_cosine_table = make_diagnostic_table(avg_results, "malicious_to_benign_mean_cosine", "raw")
raw_malicious_to_benign_mean_cosine_table


,No Attack,ALIE,MinMax,MinSum,SF,PCA k=2,PCA k=3
Defense,,,,,,,
FedAvg,,-0.6944 ± 0.0697,-0.1922 ± 0.0000,-0.2434 ± 0.0286,-1.0000 ± 0.0000,0.4786 ± 0.2703,-0.1085 ± 0.4164
Median,,-0.4765 ± 0.0731,-0.0383 ± 0.0445,-0.0298 ± 0.0306,-1.0000 ± 0.0000,0.3776 ± 0.4711,0.2651 ± 0.5328
TrimmedMean,,-0.2030 ± 0.2002,-0.0479 ± 0.0666,-0.0280 ± 0.0339,-1.0000 ± 0.0000,0.3816 ± 0.4483,0.3360 ± 0.3080
Krum,,0.0377 ± 0.0619,0.2808 ± 0.0330,-0.4510 ± 0.6265,-1.0000 ± 0.0001,0.2054 ± 0.0000,-0.1788 ± 0.0979
MultiKrum,,-0.1458 ± 0.1703,-0.0916 ± 0.2978,,-1.0000 ± 0.0000,-0.1802 ± 0.1558,-0.2211 ± 0.1184


In [141]:
nnm_malicious_to_benign_mean_cosine_table = make_diagnostic_table(avg_results, "malicious_to_benign_mean_cosine", "nnm")
nnm_malicious_to_benign_mean_cosine_table


,No Attack,ALIE,MinMax,MinSum,SF,PCA k=2,PCA k=3
Defense,,,,,,,
FedAvg,,-0.5602 ± 0.0485,-0.2436 ± 0.0920,-0.1002 ± 0.0294,-1.0000 ± 0.0000,0.5445 ± 0.3053,0.0286 ± 0.7231
Median,,-0.3385 ± 0.0378,-0.3184 ± 0.0318,0.2330 ± 0.5580,,0.4253 ± 0.1308,0.3331 ± 0.3598
TrimmedMean,,-0.3437 ± 0.0789,-0.3428 ± 0.0375,0.0701 ± 0.0273,,0.4730 ± 0.1212,0.4858 ± 0.1870
Krum,,-0.4075 ± 0.0992,-0.3309 ± 0.0275,-0.0773 ± 0.0000,-1.0000 ± 0.0000,0.3256 ± 0.1990,0.2174 ± 0.3297
MultiKrum,,-0.4017 ± 0.0754,-0.2597 ± 0.1505,0.0913 ± 0.0000,-1.0000 ± 0.0000,0.3630 ± 0.3131,0.3034 ± 0.0840


In [142]:
raw_attack_aggregation_shift_table = make_diagnostic_table(avg_results, "attack_aggregation_shift", "raw")
raw_attack_aggregation_shift_table


,No Attack,ALIE,MinMax,MinSum,SF,PCA k=2,PCA k=3
Defense,,,,,,,
FedAvg,,0.3010 ± 0.0414,0.2686 ± 0.0000,0.1516 ± 0.0430,0.0379 ± 0.0025,0.1800 ± 0.0205,0.1803 ± 0.0284
Median,,0.2415 ± 0.0956,0.1770 ± 0.0172,0.1619 ± 0.0156,0.3683 ± 0.0571,0.3078 ± 0.0356,0.3173 ± 0.0386
TrimmedMean,,0.3674 ± 0.0947,0.1555 ± 0.0128,0.1310 ± 0.0245,0.4356 ± 0.0104,0.3577 ± 0.0818,0.3249 ± 0.0419
Krum,,0.3145 ± 0.1894,0.0300 ± 0.0424,0.1701 ± 0.0473,0.3936 ± 0.2121,0.2789 ± 0.0000,0.2941 ± 0.0490
MultiKrum,,0.2755 ± 0.0676,0.3344 ± 0.2169,,0.8426 ± 1.0052,0.2285 ± 0.0159,0.2751 ± 0.0331


In [143]:
nnm_attack_aggregation_shift_table = make_diagnostic_table(avg_results, "attack_aggregation_shift", "nnm")
nnm_attack_aggregation_shift_table


,No Attack,ALIE,MinMax,MinSum,SF,PCA k=2,PCA k=3
Defense,,,,,,,
FedAvg,,0.2302 ± 0.0676,0.1446 ± 0.0470,0.1869 ± 0.0200,0.2454 ± 0.0511,0.2617 ± 0.0369,0.2453 ± 0.0349
Median,,0.3164 ± 0.0607,0.1129 ± 0.0015,0.2039 ± 0.1249,,0.3200 ± 0.0071,0.3249 ± 0.0649
TrimmedMean,,0.3029 ± 0.0441,0.0590 ± 0.0229,0.2705 ± 0.0712,,0.2426 ± 0.0323,0.2966 ± 0.0717
Krum,,0.3069 ± 0.0106,0.1278 ± 0.0456,0.2578 ± 0.0000,0.6300 ± 0.0000,0.3402 ± 0.1347,0.4275 ± 0.1082
MultiKrum,,0.3087 ± 0.0574,0.1074 ± 0.0154,0.3119 ± 0.0000,0.3232 ± 0.0000,0.3375 ± 0.0604,0.2796 ± 0.1442
